# Readmission Risk Summary — SQL

SQL-cell equivalent of `readmission_risk_summary.ipynb`, using `%%sql`
cells (window function via `LAG`) instead of the DataFrame API. Same
result: reads `medicare_claims.inpatient_claims` and persists
`medicare_claims.readmission_risk_summary_v1`.

Requires the bronze table to already exist — run
`data-generators/medicare_inpatient_claims_ingest.ipynb` first.

`%%sql` here follows Databricks-style notebook cell magic. If running
outside a platform that provides it natively, install and load
`sparksql-magic` first: `pip install sparksql-magic` then
`%load_ext sparksql_magic` (bound to the `spark` session created below).

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

## Transform + persist: readmission risk by diagnosis

A `WITH` CTE computes the prior-discharge gap per beneficiary via `LAG`,
then the outer query aggregates by primary diagnosis.

In [ ]:
%%sql
CREATE OR REPLACE TABLE medicare_claims.readmission_risk_summary_v1 AS
WITH claims_with_gap AS (
    SELECT
        ICD9_DGNS_CD_1,
        CLM_PMT_AMT,
        DATEDIFF(
            TO_DATE(CAST(CLM_ADMSN_DT AS STRING), 'yyyyMMdd'),
            TO_DATE(CAST(LAG(NCH_BENE_DSCHRG_DT) OVER (PARTITION BY DESYNPUF_ID ORDER BY CLM_ADMSN_DT) AS STRING), 'yyyyMMdd')
        ) AS days_since_prior_discharge
    FROM medicare_claims.inpatient_claims
)
SELECT
    ICD9_DGNS_CD_1 AS primary_diagnosis_code,
    COUNT(*) AS total_admissions,
    SUM(CASE WHEN days_since_prior_discharge IS NOT NULL AND days_since_prior_discharge <= 30 THEN 1 ELSE 0 END) AS readmissions_within_30d,
    ROUND(AVG(CLM_PMT_AMT), 2) AS avg_claim_payment,
    ROUND(
        SUM(CASE WHEN days_since_prior_discharge IS NOT NULL AND days_since_prior_discharge <= 30 THEN 1 ELSE 0 END) * 100.0 / COUNT(*),
        1
    ) AS readmission_rate_pct
FROM claims_with_gap
GROUP BY ICD9_DGNS_CD_1
ORDER BY total_admissions DESC

## Verify

In [ ]:
%%sql
SELECT COUNT(*) AS total_diagnoses FROM medicare_claims.readmission_risk_summary_v1

In [ ]:
%%sql
SELECT * FROM medicare_claims.readmission_risk_summary_v1
ORDER BY total_admissions DESC
LIMIT 10